In [ ]:
"""
openmc_two_slab_mg_run.py
================================
Second-stage OpenMC run using MGXS library (mgxs.h5)

- Uses precomputed MGXS (set1, set2)
- Same geometry and tallies as CE run
- Post-processing unchanged
"""

import openmc
import numpy as np
import pandas as pd


# ========================================================================
# LOAD MGXS LIBRARY
# ========================================================================
mgxs_file = openmc.MGXSLibrary.from_hdf5('mgxs.h5')

# Optional sanity check
print("Loaded MGXS domains:")
for xs in mgxs_file.xsdatas:
    print("  ", xs.name)

# ========================================================================
# MATERIALS (MG MODE)
# ========================================================================
materials = openmc.Materials()

mat1 = openmc.Material(name="slab1")
mat1.add_macroscopic("set1")   
mat1.set_density("macro", 1.0)

mat2 = openmc.Material(name="slab2")
mat2.add_macroscopic("set2")   
mat2.set_density("macro", 1.0)

materials += [mat1, mat2]
materials.cross_sections = "mgxs.h5"
materials.export_to_xml()

# ========================================================================
# GEOMETRY (UNCHANGED)
# ========================================================================
x0 = openmc.XPlane(x0=0.0,  boundary_type='vacuum')
x1 = openmc.XPlane(x0=2.0)
x2 = openmc.XPlane(x0=15.0, boundary_type='vacuum')

y0 = openmc.YPlane(y0=-1000.0, boundary_type='reflective')
y1 = openmc.YPlane(y0=1000.0, boundary_type='reflective')
z0 = openmc.ZPlane(z0=-1000.0, boundary_type='reflective')
z1 = openmc.ZPlane(z0=1000.0, boundary_type='reflective')

transverse = +y0 & -y1 & +z0 & -z1

cell1 = openmc.Cell(cell_id=15, name='slab1', fill=mat1, region=+x0 & -x1 & transverse)
cell2 = openmc.Cell(cell_id=16, name='slab2', fill=mat2, region=+x1 & -x2 & transverse)

geometry = openmc.Geometry(openmc.Universe(cells=[cell1, cell2]))
geometry.export_to_xml()

# ========================================================================
# SETTINGS (MG MODE)
# ========================================================================
N_PARTICLES = 50000
N_BATCHES   = 200

settings = openmc.Settings()
settings.run_mode  = 'fixed source'
settings.particles = N_PARTICLES
settings.batches   = N_BATCHES

# REQUIRED for MG
settings.energy_mode = "multi-group"

settings.create_fission_neutrons = False
settings.cutoff = {'energy': 10.0}

# Source (must fall inside MG group)
source = openmc.IndependentSource()
source.space    = openmc.stats.Point((1e-6, 0.0, 0.0))
source.angle    = openmc.stats.Monodirectional(reference_uvw=(1., 0., 0.))
source.energy   = openmc.stats.Discrete([500.0], [1.0])
source.particle = 'neutron'

settings.source = source
settings.export_to_xml()

# ========================================================================
# TALLIES
# ========================================================================
energy_bins = [10, 600]

cell_filter   = openmc.CellFilter([cell1, cell2])
energy_filter = openmc.EnergyFilter(energy_bins)

surf_interface = openmc.SurfaceFilter([x1])
surf_left      = openmc.SurfaceFilter([x0])
surf_right     = openmc.SurfaceFilter([x2])

# ── Flux ────────────────────────────────────────────────────────────────
t_flux = openmc.Tally(name='flux')
t_flux.filters = [cell_filter, energy_filter]
t_flux.scores  = ['flux']

# ── Reaction rates ──────────────────────────────────────────────────────
t_rxn = openmc.Tally(name='reaction_rates')
t_rxn.filters = [cell_filter, energy_filter]
t_rxn.scores  = ['absorption', 'scatter', 'fission', 'total']

# ── Currents ────────────────────────────────────────────────────────────
t_current_net = openmc.Tally(name='current_net_interface')
t_current_net.filters = [surf_interface]
t_current_net.scores  = ['current']

cellfrom_slab1 = openmc.CellFromFilter(cell1)
cellfrom_slab2 = openmc.CellFromFilter(cell2)

t_current_fwd = openmc.Tally(name='current_fwd_interface')
t_current_fwd.filters = [surf_interface, cellfrom_slab1]
t_current_fwd.scores  = ['current']

t_current_bwd = openmc.Tally(name='current_bwd_interface')
t_current_bwd.filters = [surf_interface, cellfrom_slab2]
t_current_bwd.scores  = ['current']

# ── Leakage ─────────────────────────────────────────────────────────────
t_leak_left = openmc.Tally(name='leakage_left')
t_leak_left.filters = [surf_left]
t_leak_left.scores  = ['current']

t_leak_right = openmc.Tally(name='leakage_right')
t_leak_right.filters = [surf_right]
t_leak_right.scores  = ['current']

tallies = openmc.Tallies([
    t_flux,
    t_rxn,
    t_current_net,
    t_current_fwd,
    t_current_bwd,
    t_leak_left,
    t_leak_right
])

tallies.export_to_xml()

# ========================================================================
# RUN
# ========================================================================
openmc.run()



Loaded MGXS domains:
   set1
   set2


/home/paule/anaconda3/envs/vectfit39/lib/python3.9/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=15.
  warn(msg, IDWarning)
/home/paule/anaconda3/envs/vectfit39/lib/python3.9/site-packages/openmc/mixin.py:70: IDWarning: Another Cell instance already exists with id=16.
  warn(msg, IDWarning)


                                %%%%%%%%%%%%%%%
                           %%%%%%%%%%%%%%%%%%%%%%%%
                        %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                      %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                    %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                   %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%
                                    %%%%%%%%%%%%%%%%%%%%%%%%
                                     %%%%%%%%%%%%%%%%%%%%%%%%
                 ###############      %%%%%%%%%%%%%%%%%%%%%%%%
                ##################     %%%%%%%%%%%%%%%%%%%%%%%
                ###################     %%%%%%%%%%%%%%%%%%%%%%%
                ####################     %%%%%%%%%%%%%%%%%%%%%%
                #####################     %%%%%%%%%%%%%%%%%%%%%
                ######################     %%%%%%%%%%%%%%%%%%%%
                #######################     %%%%%%%%%%%%%%%%%%
                 #######################     %%%%%%%%%%%%%%%%%
                 #####################

In [ ]:
# ========================================================================
# POST-PROCESS 
# ========================================================================
sp         = openmc.StatePoint(f'statepoint.{N_BATCHES}.h5')
volumes    = np.array([2.0, 13.0])
cell_names = ['slab1  (0–2 cm,   293.6 K)', 'slab2  (2–15 cm, 293.0 K)']
n_cells    = 2
n_groups   = len(energy_bins) - 1
e_labels   = [f"{energy_bins[i]:.0f}–{energy_bins[i+1]:.0f} eV"
              for i in range(n_groups)]

def print_table(title, units, data_mean, data_std, row_names, col_names, col_width=16):
    w = max(col_width, max(len(c) for c in col_names) + 2)
    bar = "=" * (32 + w * len(col_names))
    print(f"\n{bar}")
    print(f"  {title}")
    print(f"  Units: {units}")
    print(bar)
    print(f"  {'Region':<30}", "".join(f"{c:>{w}}" for c in col_names))
    print("-" * (32 + w * len(col_names)))
    for ri, name in enumerate(row_names):
        vals = "".join(
            f"{data_mean[ri, ei]:>{w-7}.3e} ±{data_std[ri, ei]:.1e}"
            for ei in range(len(col_names))
        )
        print(f"  {name:<30}  {vals}")
    print(bar)

# ── Flux ────────────────────────────────────────────────────────────────
t = sp.get_tally(name='flux')
flux_mean = t.get_values(scores=['flux'], value='mean').reshape(n_cells, n_groups)
flux_std  = t.get_values(scores=['flux'], value='std_dev').reshape(n_cells, n_groups)

print_table(
    'TALLY 1 — FLUX (track-length estimator)',
    'cm · source-neutron⁻¹',
    flux_mean, flux_std, cell_names, e_labels
)

flux_dens_mean = flux_mean / volumes[:, np.newaxis]
flux_dens_std  = flux_std  / volumes[:, np.newaxis]

print_table(
    'TALLY 1b — FLUX DENSITY (volume-normalised)',
    'cm⁻² · source-neutron⁻¹',
    flux_dens_mean, flux_dens_std, cell_names, e_labels
)

print("\n  FLUX SHAPE RATIO  slab1/slab2 (per unit volume):")
for ei, lab in enumerate(e_labels):
    r  = flux_dens_mean[0, ei] / flux_dens_mean[1, ei]
    print(f"    {lab:>20}  ratio = {r:.4f}")

# ── Reaction rates ──────────────────────────────────────────────────────
t = sp.get_tally(name='reaction_rates')
for score in ['absorption', 'scatter', 'fission', 'total']:
    rxn_mean = t.get_values(scores=[score], value='mean').reshape(n_cells, n_groups)
    rxn_std  = t.get_values(scores=[score], value='std_dev').reshape(n_cells, n_groups)
    print_table(
        f'TALLY 2 — {score.upper()} RATE',
        'reactions · source-neutron⁻¹',
        rxn_mean, rxn_std, cell_names, e_labels
    )

# ── Currents ────────────────────────────────────────────────────────────
t_fwd = sp.get_tally(name='current_fwd_interface')
t_bwd = sp.get_tally(name='current_bwd_interface')

fwd = float(t_fwd.get_values(scores=['current'], value='mean').flat[0])
bwd = float(t_bwd.get_values(scores=['current'], value='mean').flat[0])

print(f"\nForward current: {fwd:.4e}")
print(f"Backward current: {bwd:.4e}")
print(f"Net current: {fwd + bwd:.4e}")

# ── Leakage ─────────────────────────────────────────────────────────────
t_left  = sp.get_tally(name='leakage_left')
t_right = sp.get_tally(name='leakage_right')

leak_left  = float(t_left.get_values(scores=['current'], value='mean').flat[0])
leak_right = float(t_right.get_values(scores=['current'], value='mean').flat[0])
leak_total = -leak_left + leak_right

print(f"\nLeakage fraction = {leak_total:.4f}")

In [ ]:


rows = []

def safe_rel_err(mean, std):
    return std / mean if mean != 0 else np.inf

# ── Flux (collapsed to "all") ───────────────────────────────────────────
t = sp.get_tally(name='flux')
flux_mean = t.get_values(scores=['flux'], value='mean').reshape(n_cells, n_groups)
flux_std  = t.get_values(scores=['flux'], value='std_dev').reshape(n_cells, n_groups)

# sum over regions → "all"
flux_mean_all = flux_mean.sum(axis=0)
flux_std_all  = np.sqrt((flux_std**2).sum(axis=0))  # proper error propagation

for ei, energy in enumerate(e_labels):
    mean = flux_mean_all[ei]
    std  = flux_std_all[ei]
    rows.append({
        "tally": "flux",
        "region": "all",
        "energy_group": energy,
        "mean": mean,
        "std": std,
        "relative_error": safe_rel_err(mean, std)
    })

# ── Reaction rates (keep regions) ───────────────────────────────────────
t = sp.get_tally(name='reaction_rates')

region_labels = ['0.0-2.0 cm', '2.0-15.0 cm']

for score in ['absorption', 'scatter']:
    rxn_mean = t.get_values(scores=[score], value='mean').reshape(n_cells, n_groups)
    rxn_std  = t.get_values(scores=[score], value='std_dev').reshape(n_cells, n_groups)

    for ri, region in enumerate(region_labels):
        for ei, energy in enumerate(e_labels):
            mean = rxn_mean[ri, ei]
            std  = rxn_std[ri, ei]

            rows.append({
                "tally": score,
                "region": region,
                "energy_group": energy,
                "mean": mean,
                "std": std,
                "relative_error": safe_rel_err(mean, std)
            })

# ── Currents (single values, energy = "all") ─────────────────────────────
t_fwd = sp.get_tally(name='current_fwd_interface')
t_bwd = sp.get_tally(name='current_bwd_interface')

fwd_mean = float(t_fwd.get_values(scores=['current'], value='mean'))
fwd_std  = float(t_fwd.get_values(scores=['current'], value='std_dev'))

bwd_mean = float(t_bwd.get_values(scores=['current'], value='mean'))
bwd_std  = float(t_bwd.get_values(scores=['current'], value='std_dev'))

rows.append({
    "tally": "current_forward",
    "region": "x=2.00 cm",
    "energy_group": "all",
    "mean": fwd_mean,
    "std": fwd_std,
    "relative_error": safe_rel_err(fwd_mean, fwd_std)
})

rows.append({
    "tally": "current_backward",
    "region": "x=2.00 cm",
    "energy_group": "all",
    "mean": bwd_mean,
    "std": bwd_std,
    "relative_error": safe_rel_err(bwd_mean, bwd_std)
})

# ── Leakage ─────────────────────────────────────────────────────────────
t_left  = sp.get_tally(name='leakage_left')
t_right = sp.get_tally(name='leakage_right')

left_mean = float(t_left.get_values(scores=['current'], value='mean'))
left_std  = float(t_left.get_values(scores=['current'], value='std_dev'))

right_mean = float(t_right.get_values(scores=['current'], value='mean'))
right_std  = float(t_right.get_values(scores=['current'], value='std_dev'))

rows.append({
    "tally": "leak_left",
    "region": "boundary",
    "energy_group": "all",
    "mean": left_mean,
    "std": left_std,
    "relative_error": safe_rel_err(left_mean, left_std)
})

rows.append({
    "tally": "leak_right",
    "region": "boundary",
    "energy_group": "all",
    "mean": right_mean,
    "std": right_std,
    "relative_error": safe_rel_err(right_mean, right_std)
})

# ── DataFrame + export ──────────────────────────────────────────────────
df = pd.DataFrame(rows, columns=[
    "tally",
    "region",
    "energy_group",
    "mean",
    "std",
    "relative_error"
])

df.to_csv("openmc_results.csv", index=False)

print("CSV written: openmc_results.csv")

CSV written: openmc_results.csv


/tmp/ipykernel_58491/2876732333.py:57: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  fwd_mean = float(t_fwd.get_values(scores=['current'], value='mean'))
/tmp/ipykernel_58491/2876732333.py:58: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  fwd_std  = float(t_fwd.get_values(scores=['current'], value='std_dev'))
/tmp/ipykernel_58491/2876732333.py:60: DeprecationWarning: Conversion of an array with ndim > 0 to a scalar is deprecated, and will error in future. Ensure you extract a single element from your array before performing this operation. (Deprecated NumPy 1.25.)
  bwd_mean = float(t_bwd.get_values(scores=['current'], value='mean'))
/tmp